# RetailMind AI — Phase 4: Demand Forecasting

**Phase:** 4 — Demand Forecasting  
**Stage:** 1 — Data Loading and Verification

---

## Objective

Develop demand forecasting models using the engineered daily product demand dataset produced in Phase 3 — Feature Engineering.

| Property | Value |
|---|---|
| **Target variable** | `DailyQuantity` — total units sold for a product on a given date |
| **Granularity** | `ProductID` × `Date` |
| **Evaluation metrics** | MAE, RMSE, MAPE *(implemented in a later stage)* |
| **Split strategy** | Time-based train / validation / test split *(implemented in a later stage)* |

---

## Stage 1 Scope

This stage covers **data loading and verification only**.

The following steps are deferred to later stages:
- Feature selection for forecasting
- Handling lag/rolling NaN values appropriately
- Time-based train/validation/test split
- Model training and evaluation

**Prerequisites:**
- `01_data_understanding.ipynb` — Data quality validation ✅
- `02_eda.ipynb` — Exploratory analysis ✅
- `03_feature_engineering.ipynb` — Feature engineering ✅

**Input dataset:**
`data/processed/daily_product_demand.csv`

---
## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded successfully.')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

Libraries loaded successfully.
  pandas  : 3.0.5
  numpy   : 2.4.0


---
## Cell 2 — Dataset Path

In [2]:
DEMAND_PATH = '../data/processed/daily_product_demand.csv'

# Confirm file exists before loading
if os.path.exists(DEMAND_PATH):
    size_kb = os.path.getsize(DEMAND_PATH) / 1024
    print(f'Dataset found: {os.path.abspath(DEMAND_PATH)}')
    print(f'File size    : {size_kb:,.1f} KB')
else:
    raise FileNotFoundError(
        f'STOPPING: Dataset not found at {DEMAND_PATH}. '
        'Run 03_feature_engineering.ipynb first.'
    )

Dataset found: D:\RetailMindAI\RetailMindAI\data\processed\daily_product_demand.csv
File size    : 43,391.3 KB


---
## Cell 3 — Load Dataset

> The dataset is loaded **as-is** from Phase 3. No rows are dropped, no NaN values are filled, and no transformations are applied at this stage.

In [3]:
demand = pd.read_csv(DEMAND_PATH)

# Convert Date to datetime immediately after loading
demand['Date'] = pd.to_datetime(demand['Date'])

print('Dataset loaded successfully.')
print(f'Shape: {demand.shape[0]:,} rows x {demand.shape[1]} columns')

Dataset loaded successfully.
Shape: 91,250 rows x 48 columns


---
## Cell 4 — Dataset Verification

Print key characteristics of the loaded dataset to confirm it matches the expected Phase 3 output.

In [4]:
print('=' * 55)
print('  DATASET VERIFICATION')
print('=' * 55)

# Rows and columns
print(f'  Rows               : {demand.shape[0]:,}')
print(f'  Columns            : {demand.shape[1]}')

# Date range
date_min = demand['Date'].min()
date_max = demand['Date'].max()
n_dates  = demand['Date'].nunique()
print(f'  Date range         : {date_min.date()} to {date_max.date()}')
print(f'  Unique dates       : {n_dates:,}')

# Products
n_products = demand['ProductID'].nunique()
print(f'  Unique products    : {n_products}')

# Target column
target_col = 'DailyQuantity'
target_present = target_col in demand.columns
print(f'  Target column      : {target_col}  (present: {target_present})')

# Missing values in target
target_missing = demand[target_col].isna().sum()
print(f'  Missing in target  : {target_missing}')

print('=' * 55)

  DATASET VERIFICATION
  Rows               : 91,250
  Columns            : 48
  Date range         : 2020-01-01 to 2024-12-29
  Unique dates       : 1,825
  Unique products    : 50
  Target column      : DailyQuantity  (present: True)
  Missing in target  : 0


---
## Cell 5 — Column Overview

In [5]:
print(f'All columns ({demand.shape[1]} total):')
for i, col in enumerate(demand.columns, 1):
    dtype   = str(demand[col].dtype)
    n_miss  = demand[col].isna().sum()
    miss_pct = n_miss / len(demand) * 100
    marker  = '  <- TARGET' if col == 'DailyQuantity' else ''
    print(f'  {i:2d}. {col:<35} dtype={dtype:<10} missing={n_miss:,} ({miss_pct:.2f}%){marker}')

All columns (48 total):
   1. ProductID                           dtype=str        missing=0 (0.00%)
   2. Date                                dtype=datetime64[us] missing=0 (0.00%)
   3. DailyQuantity                       dtype=float64    missing=0 (0.00%)  <- TARGET
   4. DailyRevenue                        dtype=float64    missing=0 (0.00%)
   5. DailyOrderCount                     dtype=float64    missing=0 (0.00%)
   6. DailyAveragePrice                   dtype=float64    missing=0 (0.00%)
   7. DailyAverageDiscount                dtype=float64    missing=0 (0.00%)
   8. DailyReturnCount                    dtype=float64    missing=0 (0.00%)
   9. DailyCancellationCount              dtype=float64    missing=0 (0.00%)
  10. DailyAverageOrderValue              dtype=float64    missing=0 (0.00%)
  11. ProductName                         dtype=str        missing=0 (0.00%)
  12. year                                dtype=int64      missing=0 (0.00%)
  13. month                          

---
## Cell 6 — Sample Rows

Display a small sample to visually confirm the dataset structure.

In [6]:
display_cols = [
    'ProductID', 'Date', 'DailyQuantity',
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28',
    'is_weekend', 'month_number', 'year'
]
print('Sample: first 10 rows of product P00001')
sample = demand[demand['ProductID'] == demand['ProductID'].iloc[0]][display_cols].head(10)
display(sample)

Sample: first 10 rows of product P00001


,ProductID,Date,DailyQuantity,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,is_weekend,month_number,year
0,P00001,2020-01-01,2.0000,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2020
1,P00001,2020-01-02,0.0000,2.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
2,P00001,2020-01-03,5.0000,0.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
3,P00001,2020-01-04,3.0000,5.0000,NaN,NaN,NaN,NaN,NaN,1,1,2020
4,P00001,2020-01-05,1.0000,3.0000,NaN,NaN,NaN,NaN,NaN,1,1,2020
5,P00001,2020-01-06,2.0000,1.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
6,P00001,2020-01-07,0.0000,2.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
7,P00001,2020-01-08,7.0000,0.0000,2.0000,NaN,NaN,1.8571,NaN,0,1,2020
8,P00001,2020-01-09,3.0000,7.0000,0.0000,NaN,NaN,2.5714,NaN,0,1,2020
9,P00001,2020-01-10,5.0000,3.0000,5.0000,NaN,NaN,3.0000,NaN,0,1,2020


---
## Cell 7 — Lag / Rolling NaN Summary

> **Important:** The NaN values shown below in lag and rolling features are **expected and correct**. They arise because the first N days of each product's history cannot have a valid N-day lookback.

**These NaN values must NOT be filled or dropped at this stage.** They will be handled appropriately during feature selection and train/test split preparation in the next stage.

In [7]:
lag_rolling_cols = [
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_std_7',
    'rolling_mean_14', 'rolling_std_14',
    'rolling_mean_28', 'rolling_std_28',
    'short_term_mean', 'medium_term_mean', 'long_term_mean',
    'short_vs_medium_growth', 'medium_vs_long_growth',
    'cv_7', 'cv_28',
    'rolling_return_rate_28', 'rolling_cancel_rate_28',
    'rolling_revenue_mean_28', 'rolling_revenue_std_28',
    'revenue_growth_7_vs_28'
]

n_products = demand['ProductID'].nunique()
print('Lag/Rolling NaN Summary (all expected):')
print(f'  (Based on {n_products} products with complete daily grid)')
print()
for col in lag_rolling_cols:
    if col in demand.columns:
        n_nan = demand[col].isna().sum()
        pct   = n_nan / len(demand) * 100
        print(f'  {col:<30} : {n_nan:,} NaN ({pct:.2f}%)')
print()
print('NOTE: Do NOT fill or drop these NaN values at this stage.')

Lag/Rolling NaN Summary (all expected):
  (Based on 50 products with complete daily grid)

  lag_1                          : 50 NaN (0.05%)
  lag_7                          : 350 NaN (0.38%)
  lag_14                         : 700 NaN (0.77%)
  lag_28                         : 1,400 NaN (1.53%)
  rolling_mean_7                 : 350 NaN (0.38%)
  rolling_std_7                  : 350 NaN (0.38%)
  rolling_mean_14                : 700 NaN (0.77%)
  rolling_std_14                 : 700 NaN (0.77%)
  rolling_mean_28                : 1,400 NaN (1.53%)
  rolling_std_28                 : 1,400 NaN (1.53%)
  short_term_mean                : 350 NaN (0.38%)
  medium_term_mean               : 1,400 NaN (1.53%)
  long_term_mean                 : 4,500 NaN (4.93%)
  short_vs_medium_growth         : 1,400 NaN (1.53%)
  medium_vs_long_growth          : 4,500 NaN (4.93%)
  cv_7                           : 396 NaN (0.43%)
  cv_28                          : 1,400 NaN (1.53%)
  rolling_return_rate_28   

---
## Cell 8 — Expected Phase 3 Characteristics Check

Assert that the dataset matches the documented Phase 3 output exactly.

In [8]:
print('=' * 55)
print('  PHASE 3 CHARACTERISTICS CHECK')
print('=' * 55)

checks = []

# Check 1: row count
actual_rows = demand.shape[0]
row_ok = (actual_rows == 91250)
checks.append(('Rows == 91,250', row_ok, f'actual={actual_rows:,}'))

# Check 2: unique products
actual_prods = demand['ProductID'].nunique()
prod_ok = (actual_prods == 50)
checks.append(('Unique products == 50', prod_ok, f'actual={actual_prods}'))

# Check 3: start date
actual_start = demand['Date'].min().date()
from datetime import date
start_ok = (actual_start == date(2020, 1, 1))
checks.append(('Start date == 2020-01-01', start_ok, f'actual={actual_start}'))

# Check 4: end date
actual_end = demand['Date'].max().date()
end_ok = (actual_end == date(2024, 12, 29))
checks.append(('End date == 2024-12-29', end_ok, f'actual={actual_end}'))

# Check 5: DailyQuantity present
target_ok = ('DailyQuantity' in demand.columns)
checks.append(('DailyQuantity column exists', target_ok, ''))

# Check 6: DailyQuantity has no missing values
target_miss = demand['DailyQuantity'].isna().sum()
target_miss_ok = (target_miss == 0)
checks.append(('DailyQuantity has 0 missing', target_miss_ok, f'actual={target_miss}'))

# Check 7: no duplicate ProductID-Date pairs
n_dup = demand.duplicated(subset=['ProductID', 'Date']).sum()
dup_ok = (n_dup == 0)
checks.append(('No duplicate ProductID-Date rows', dup_ok, f'duplicates={n_dup}'))

all_passed = True
for label, passed, detail in checks:
    status = 'PASS' if passed else 'FAIL'
    detail_str = f'  ({detail})' if detail else ''
    print(f'  [{status}] {label}{detail_str}')
    if not passed:
        all_passed = False

print('=' * 55)
print(f"  OVERALL: {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")
print('=' * 55)

  PHASE 3 CHARACTERISTICS CHECK
  [PASS] Rows == 91,250  (actual=91,250)
  [PASS] Unique products == 50  (actual=50)
  [PASS] Start date == 2020-01-01  (actual=2020-01-01)
  [PASS] End date == 2024-12-29  (actual=2024-12-29)
  [PASS] DailyQuantity column exists
  [PASS] DailyQuantity has 0 missing  (actual=0)
  [PASS] No duplicate ProductID-Date rows  (duplicates=0)
  OVERALL: ALL CHECKS PASSED


---
## Cell 9 — Stage 1 Summary

In [9]:
print('=' * 55)
print('  STAGE 1 COMPLETE — Data Loading & Verification')
print('=' * 55)

print('  Dataset loaded : data/processed/daily_product_demand.csv')
print(f'  Rows           : {demand.shape[0]:,}')
print(f'  Columns        : {demand.shape[1]}')
print(f'  Products       : {demand["ProductID"].nunique()}')
print(f'  Date range     : {demand["Date"].min().date()} to {demand["Date"].max().date()}')
print(f'  Target         : DailyQuantity  (0 missing values)')
print(f'  Dataset status : READY for Phase 4 Stage 2')

print()
print('  NEXT STAGE (implement separately):')
print('  - Select forecasting features from the 48 available columns')
print('  - Handle lag/rolling NaN values (drop or mask for training)')
print('  - Apply time-based train/validation/test split')
print('  - Train demand forecasting model')
print('  - Evaluate with MAE, RMSE, MAPE')

print('=' * 55)
print('  Do NOT proceed into model training in this notebook.')
print('=' * 55)

  STAGE 1 COMPLETE — Data Loading & Verification
  Dataset loaded : data/processed/daily_product_demand.csv
  Rows           : 91,250
  Columns        : 48
  Products       : 50
  Date range     : 2020-01-01 to 2024-12-29
  Target         : DailyQuantity  (0 missing values)
  Dataset status : READY for Phase 4 Stage 2

  NEXT STAGE (implement separately):
  - Select forecasting features from the 48 available columns
  - Handle lag/rolling NaN values (drop or mask for training)
  - Apply time-based train/validation/test split
  - Train demand forecasting model
  - Evaluate with MAE, RMSE, MAPE
  Do NOT proceed into model training in this notebook.


---

# Stage 2 — Feature Selection, Split & Model Training

**Building on Stage 1:** The dataset is already loaded and verified.

**Stage 2 covers:**
1. Feature selection — choose model inputs from the 48 engineered columns
2. NaN handling — drop warm-up rows with insufficient lag/rolling history
3. Time-based train / validation / test split (never random shuffle)
4. Baseline model training — LightGBM, XGBoost, and Linear Regression
5. Evaluation — MAE, RMSE, MAPE per model and per product
6. Feature importance

---
## Cell 10 — Extended Imports for Stage 2

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

print('All Stage 2 libraries loaded.')
print(f'  scikit-learn : {__import__("sklearn").__version__}')
print(f'  lightgbm     : {lgb.__version__}')
print(f'  xgboost      : {xgb.__version__}')

All Stage 2 libraries loaded.
  scikit-learn : 1.8.0
  lightgbm     : 4.7.0
  xgboost      : 3.4.0


---
## Cell 11 — Feature Selection

### Selection criteria

**Included — safe lag/rolling demand history:**
- `lag_1`, `lag_7`, `lag_14`, `lag_28` — direct autoregressive signal
- `rolling_mean_7`, `rolling_mean_14`, `rolling_mean_28` — smoothed recent demand
- `rolling_std_7`, `rolling_std_28` — local demand volatility
- `short_term_mean`, `medium_term_mean`, `long_term_mean` — trend context
- `short_vs_medium_growth`, `medium_vs_long_growth` — growth signals
- `cv_7`, `cv_28` — relative volatility (Coefficient of Variation)

**Included — leakage-safe business signals:**
- `rolling_return_rate_28`, `rolling_cancel_rate_28` — quality signals
- `rolling_revenue_mean_28`, `revenue_growth_7_vs_28` — revenue context

**Included — temporal calendar features (no leakage):**
- `month_number`, `quarter`, `week_of_year`
- `day_of_week_num`, `day_of_month`
- `is_weekend`, `is_month_start`, `is_month_end`
- `year`

**Excluded — target-adjacent (would cause leakage in a live system):**
- `DailyRevenue`, `DailyOrderCount`, `DailyReturnCount`, `DailyCancellationCount`
- `DailyAveragePrice`, `DailyAverageDiscount`, `DailyAverageOrderValue`
- `daily_return_rate`, `daily_cancel_rate` *(raw daily — use rolling instead)*

**Excluded — identifiers/metadata:**
- `ProductID` (used for grouping only), `ProductName`, `Date`
- `month` (string — encoded as `month_number`)

In [11]:
TARGET = 'DailyQuantity'

FEATURE_COLS = [
    # Lag features
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    # Rolling mean/std
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
    'rolling_std_7', 'rolling_std_28',
    # Trend features
    'short_term_mean', 'medium_term_mean', 'long_term_mean',
    'short_vs_medium_growth', 'medium_vs_long_growth',
    # Volatility
    'cv_7', 'cv_28',
    # Risk signals (rolling = no leakage)
    'rolling_return_rate_28', 'rolling_cancel_rate_28',
    # Revenue signals (rolling = no leakage)
    'rolling_revenue_mean_28', 'revenue_growth_7_vs_28',
    # Temporal
    'year', 'month_number', 'quarter', 'week_of_year',
    'day_of_week_num', 'day_of_month',
    'is_weekend', 'is_month_start', 'is_month_end',
]

print(f'Target column  : {TARGET}')
print(f'Feature columns: {len(FEATURE_COLS)}')
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {col}')

Target column  : DailyQuantity
Feature columns: 29
   1. lag_1
   2. lag_7
   3. lag_14
   4. lag_28
   5. rolling_mean_7
   6. rolling_mean_14
   7. rolling_mean_28
   8. rolling_std_7
   9. rolling_std_28
  10. short_term_mean
  11. medium_term_mean
  12. long_term_mean
  13. short_vs_medium_growth
  14. medium_vs_long_growth
  15. cv_7
  16. cv_28
  17. rolling_return_rate_28
  18. rolling_cancel_rate_28
  19. rolling_revenue_mean_28
  20. revenue_growth_7_vs_28
  21. year
  22. month_number
  23. quarter
  24. week_of_year
  25. day_of_week_num
  26. day_of_month
  27. is_weekend
  28. is_month_start
  29. is_month_end


---
## Cell 12 — NaN Handling Strategy

### Why NaNs exist
Lag and rolling features cannot be computed for the first N days of each product's history (warm-up period). The largest lookback is `long_term_mean` at 90 days — so the first 90 rows per product have at least one NaN.

### Strategy: Drop warm-up rows
Drop any row where **any selected feature has a NaN value**. This removes exactly the warm-up period at the start of each product's history.

> This is correct for training and evaluation. In a live forecasting system, the model would be applied only after sufficient history has accumulated.

In [12]:
# Build the modelling-ready subset
model_df = demand[['ProductID', 'Date', TARGET] + FEATURE_COLS].copy()

before = len(model_df)
model_df = model_df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
after  = len(model_df)
dropped = before - after

print('=== NaN Handling ===')
print(f'  Rows before dropna : {before:,}')
print(f'  Rows dropped       : {dropped:,}  ({dropped/before*100:.2f}%)')
print(f'  Rows after dropna  : {after:,}')
print(f'  Missing in features: {model_df[FEATURE_COLS].isnull().sum().sum()}')
print(f'  Missing in target  : {model_df[TARGET].isnull().sum()}')
print()
print(f'  Date range retained: {model_df["Date"].min().date()} to {model_df["Date"].max().date()}')
print(f'  Products retained  : {model_df["ProductID"].nunique()}')

=== NaN Handling ===
  Rows before dropna : 91,250
  Rows dropped       : 4,536  (4.97%)
  Rows after dropna  : 86,714
  Missing in features: 0
  Missing in target  : 0

  Date range retained: 2020-03-31 to 2024-12-29
  Products retained  : 50


---
## Cell 13 — Time-Based Train / Validation / Test Split

### Why time-based split?
Random shuffling would mix future data into training — introducing temporal leakage that inflates model performance. A time-based split keeps the chronological order intact.

### Split boundaries

| Split | Date Range | Approx. Share |
|---|---|---|
| **Train** | 2020-01-01 → 2023-06-30 | ~70% |
| **Validation** | 2023-07-01 → 2024-03-31 | ~15% |
| **Test** | 2024-04-01 → 2024-12-29 | ~15% |

> The test set contains only 2024 data — a realistic held-out evaluation period.

In [13]:
TRAIN_END = pd.Timestamp('2023-06-30')
VAL_END   = pd.Timestamp('2024-03-31')

train_df = model_df[model_df['Date'] <= TRAIN_END].copy()
val_df   = model_df[(model_df['Date'] > TRAIN_END) & (model_df['Date'] <= VAL_END)].copy()
test_df  = model_df[model_df['Date'] > VAL_END].copy()

X_train, y_train = train_df[FEATURE_COLS], train_df[TARGET]
X_val,   y_val   = val_df[FEATURE_COLS],   val_df[TARGET]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df[TARGET]

print('=== Time-Based Split ===')
print(f'  Train      : {train_df["Date"].min().date()} to {train_df["Date"].max().date()}  |  {len(train_df):,} rows')
print(f'  Validation : {val_df["Date"].min().date()}   to {val_df["Date"].max().date()}   |  {len(val_df):,} rows')
print(f'  Test       : {test_df["Date"].min().date()}   to {test_df["Date"].max().date()}  |  {len(test_df):,} rows')
print()
print(f'  X_train shape: {X_train.shape}')
print(f'  X_val shape  : {X_val.shape}')
print(f'  X_test shape : {X_test.shape}')
print()
print('Chronological order check:')
print(f'  max(train) < min(val) : {train_df["Date"].max() < val_df["Date"].min()}')
print(f'  max(val)   < min(test): {val_df["Date"].max()   < test_df["Date"].min()}')

=== Time-Based Split ===
  Train      : 2020-03-31 to 2023-06-30  |  59,324 rows
  Validation : 2023-07-01   to 2024-03-31   |  13,747 rows
  Test       : 2024-04-01   to 2024-12-29  |  13,643 rows

  X_train shape: (59324, 29)
  X_val shape  : (13747, 29)
  X_test shape : (13643, 29)

Chronological order check:
  max(train) < min(val) : True
  max(val)   < min(test): True


---
## Cell 14 — Evaluation Metric Functions

| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** | `mean(|y - ŷ|)` | Average absolute error in units |
| **RMSE** | `sqrt(mean((y - ŷ)²))` | Penalises large errors more heavily |
| **MAPE** | `mean(|y - ŷ| / y) × 100` | Percentage error (excludes zero-demand days) |

In [14]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error. Excludes zero-demand rows."""
    mask = y_true != 0
    if mask.sum() == 0:
        return float("nan")
    return (abs(y_true[mask] - y_pred[mask]) / y_true[mask]).mean() * 100


def evaluate(name, y_true, y_pred):
    """Compute and print MAE, RMSE, MAPE."""
    import numpy as np
    mae_  = mean_absolute_error(y_true, y_pred)
    rmse_ = mean_squared_error(y_true, y_pred) ** 0.5
    mape_ = mape(y_true.values, y_pred)
    print(f"  {name:<20} MAE={mae_:6.3f}  RMSE={rmse_:6.3f}  MAPE={mape_:6.2f}%")
    return {"Model": name, "MAE": round(mae_,3), "RMSE": round(rmse_,3), "MAPE": round(mape_,2)}


print('Metric functions defined: MAE, RMSE, MAPE')

Metric functions defined: MAE, RMSE, MAPE


---
## Cell 15 — Model Training

### Models trained

| Model | Rationale |
|---|---|
| **LightGBM** | Gradient boosting — strong on tabular time-series features, handles NaN internally |
| **XGBoost** | Gradient boosting — robust benchmark, widely used in retail forecasting |
| **Ridge Regression** | Linear baseline — establishes a minimum performance floor |

> All models use the same feature set and the same time-based split.

> **No hyperparameter tuning yet** — default/conservative parameters only.

In [15]:
results_val  = []
results_test = []
trained_models = {}

print('=== Model Training and Evaluation ===')
print(f'  Train rows : {len(X_train):,}')
print(f'  Val rows   : {len(X_val):,}')
print(f'  Test rows  : {len(X_test):,}')
print()

# ── 1. LightGBM ────────────────────────────────────────────────────────────
print('--- LightGBM ---')
lgb_model = lgb.LGBMRegressor(
    n_estimators   = 500,
    learning_rate  = 0.05,
    num_leaves     = 31,
    min_child_samples = 20,
    random_state   = 42,
    verbose        = -1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)
lgb_pred_val  = lgb_model.predict(X_val)
lgb_pred_test = lgb_model.predict(X_test)
results_val.append(evaluate('LightGBM',  y_val,  lgb_pred_val))
results_test.append(evaluate('LightGBM', y_test, lgb_pred_test))
trained_models['LightGBM'] = lgb_model
print()

# ── 2. XGBoost ─────────────────────────────────────────────────────────────
print('--- XGBoost ---')
xgb_model = xgb.XGBRegressor(
    n_estimators  = 500,
    learning_rate = 0.05,
    max_depth     = 6,
    subsample     = 0.8,
    colsample_bytree = 0.8,
    random_state  = 42,
    verbosity     = 0,
    early_stopping_rounds = 50
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
xgb_pred_val  = xgb_model.predict(X_val)
xgb_pred_test = xgb_model.predict(X_test)
results_val.append(evaluate('XGBoost',  y_val,  xgb_pred_val))
results_test.append(evaluate('XGBoost', y_test, xgb_pred_test))
trained_models['XGBoost'] = xgb_model
print()

# ── 3. Ridge (linear baseline) ─────────────────────────────────────────────
print('--- Ridge Regression (linear baseline) ---')
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_sc, y_train)
ridge_pred_val  = ridge_model.predict(X_val_sc)
ridge_pred_test = ridge_model.predict(X_test_sc)
results_val.append(evaluate('Ridge',  y_val,  ridge_pred_val))
results_test.append(evaluate('Ridge', y_test, ridge_pred_test))
trained_models['Ridge'] = ridge_model

=== Model Training and Evaluation ===
  Train rows : 59,324
  Val rows   : 13,747
  Test rows  : 13,643

--- LightGBM ---


  LightGBM             MAE= 2.808  RMSE= 3.485  MAPE= 61.59%
  LightGBM             MAE= 2.821  RMSE= 3.494  MAPE= 61.99%

--- XGBoost ---


  XGBoost              MAE= 2.808  RMSE= 3.484  MAPE= 61.60%
  XGBoost              MAE= 2.821  RMSE= 3.494  MAPE= 62.00%

--- Ridge Regression (linear baseline) ---


  Ridge                MAE= 2.808  RMSE= 3.486  MAPE= 61.59%
  Ridge                MAE= 2.819  RMSE= 3.495  MAPE= 61.87%


---
## Cell 16 — Evaluation Results Summary

In [16]:
print('=== Validation Set Results ===')
val_results_df = pd.DataFrame(results_val).set_index("Model")
display(val_results_df)
print()

print('=== Test Set Results (Hold-out — 2024-04-01 to 2024-12-29) ===')
test_results_df = pd.DataFrame(results_test).set_index("Model")
display(test_results_df)
print()

# Identify best model by test MAE
best_model_name = test_results_df['MAE'].idxmin()
print(f'Best model by test MAE: {best_model_name}')
print(f'  MAE  = {test_results_df.loc[best_model_name, "MAE"]}')
print(f'  RMSE = {test_results_df.loc[best_model_name, "RMSE"]}')
print(f'  MAPE = {test_results_df.loc[best_model_name, "MAPE"]}%')

=== Validation Set Results ===


,MAE,RMSE,MAPE
Model,,,
LightGBM,2.8080,3.4850,61.5900
XGBoost,2.8080,3.4840,61.6000
Ridge,2.8080,3.4860,61.5900



=== Test Set Results (Hold-out — 2024-04-01 to 2024-12-29) ===


,MAE,RMSE,MAPE
Model,,,
LightGBM,2.8210,3.4940,61.9900
XGBoost,2.8210,3.4940,62.0000
Ridge,2.8190,3.4950,61.8700



Best model by test MAE: Ridge
  MAE  = 2.819
  RMSE = 3.495
  MAPE = 61.87%


---
## Cell 17 — Per-Product Test Set Evaluation (LightGBM)

Evaluate the best-performing tree model on each of the 50 products individually. This reveals which products are harder to forecast and may need product-specific treatment.

In [17]:
test_df_eval = test_df[['ProductID', 'Date', TARGET] + FEATURE_COLS].copy()
test_df_eval['lgb_pred'] = lgb_model.predict(test_df_eval[FEATURE_COLS])

per_product = []
for pid, grp in test_df_eval.groupby('ProductID'):
    yt = grp[TARGET]
    yp = grp['lgb_pred']
    per_product.append({
        'ProductID' : pid,
        'MAE'       : round(mean_absolute_error(yt, yp), 3),
        'RMSE'      : round(mean_squared_error(yt, yp)**0.5, 3),
        'MAPE'      : round(mape(yt.values, yp.values), 2),
        'TestRows'  : len(grp)
    })

pp_df = pd.DataFrame(per_product).sort_values('MAE', ascending=False).reset_index(drop=True)

print(f'Per-product LightGBM Test MAE (sorted worst to best):')
display(pp_df)

print(f'\nMean MAE  across products: {pp_df["MAE"].mean():.3f}')
print(f'Median MAE across products: {pp_df["MAE"].median():.3f}')
print(f'Max MAE (hardest product) : {pp_df["MAE"].max():.3f}  -> {pp_df.iloc[0]["ProductID"]}')
print(f'Min MAE (easiest product) : {pp_df["MAE"].min():.3f}  -> {pp_df.loc[pp_df["MAE"].idxmin(), "ProductID"]}')

Per-product LightGBM Test MAE (sorted worst to best):

,ProductID,MAE,RMSE,MAPE,TestRows
0,P00008,3.0280,3.8230,62.1100,273
1,P00034,2.9970,3.7460,69.7300,273
2,P00047,2.9890,3.6950,64.3100,273
3,P00022,2.9820,3.9330,54.0600,273
4,P00042,2.9800,3.6840,70.2100,273
5,P00048,2.9780,3.7360,63.1800,273
6,P00050,2.9520,3.6570,62.8700,273
7,P00017,2.9490,3.6640,71.4500,273
8,P00018,2.9480,3.6140,59.4700,273
9,P00037,2.9430,3.6820,68.0800,273



Mean MAE  across products: 2.821
Median MAE across products: 2.817
Max MAE (hardest product) : 3.028  -> P00008
Min MAE (easiest product) : 2.428  -> P00049


---
## Cell 18 — LightGBM Feature Importance

Feature importance shows which engineered features contribute most to demand prediction.

In [18]:
importance_df = pd.DataFrame({
    'Feature'    : FEATURE_COLS,
    'Importance' : lgb_model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('LightGBM Feature Importance (top 20):')
display(importance_df.head(20))

# Save figure
fig, ax = plt.subplots(figsize=(9, 7))
top20 = importance_df.head(20)
ax.barh(top20['Feature'][::-1], top20['Importance'][::-1], color='#4E79A7')
ax.set_xlabel('Importance (split gain)')
ax.set_title('LightGBM — Top 20 Feature Importances')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
fig_path = '../outputs/figures/lgb_feature_importance.png'
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.close()
print(f'Figure saved: {fig_path}')

LightGBM Feature Importance (top 20):


,Feature,Importance
0,revenue_growth_7_vs_28,4
1,week_of_year,3
2,cv_28,3
3,rolling_return_rate_28,3
4,short_vs_medium_growth,3
5,rolling_std_28,2
6,rolling_std_7,2
7,day_of_month,2
8,lag_14,1
9,long_term_mean,1


Figure saved: ../outputs/figures/lgb_feature_importance.png


---
## Cell 19 — Stage 2 Summary

In [19]:
print('=' * 60)
print('  PHASE 4 STAGE 2 COMPLETE — Demand Forecasting')
print('=' * 60)
print()
print('  Feature columns   :', len(FEATURE_COLS))
print('  NaN rows dropped  :', before - after, '(warm-up period)')
print('  Modelling rows    :', after)
print('  Train / Val / Test:', len(train_df), '/', len(val_df), '/', len(test_df))
print()
print('  Models trained    : LightGBM, XGBoost, Ridge')
print('  Best test MAE     :', test_results_df['MAE'].min(), f'({best_model_name})')
print()
print('  NEXT STEPS (separate tasks):')
print('  - Hyperparameter tuning (Optuna / grid search)')
print('  - Product-level models for high-error products')
print('  - Conflict detection module (Phase 5)')
print('=' * 60)

  PHASE 4 STAGE 2 COMPLETE — Demand Forecasting

  Feature columns   : 29
  NaN rows dropped  : 4536 (warm-up period)
  Modelling rows    : 86714
  Train / Val / Test: 59324 / 13747 / 13643

  Models trained    : LightGBM, XGBoost, Ridge
  Best test MAE     : 2.819 (Ridge)

  NEXT STEPS (separate tasks):
  - Hyperparameter tuning (Optuna / grid search)
  - Product-level models for high-error products
  - Conflict detection module (Phase 5)
